In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from viz_config import VizConfig

# ==========================================
# 0. Global configuration and environment setup
# ==========================================
# Load the shared plotting style (viz_config.py) so every figure uses the same fonts and colours
VizConfig.set_style()

# Define the output directory for the generated PDFs
OUTPUT_DIR = "Stage2_Hypothesis_Verification"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Concentration scaling factor - avoids numerical instability caused by very small values
# Raw values are ~1e-7; after scaling they become ~1.0
SCALE = 1e7

# ==========================================
# 1. Data preparation
# ==========================================
print("Reading data...")
if not os.path.exists('data/train_dataset_ready.csv'):
    print("ERROR: train_dataset_ready.csv not found")
    exit()

# Read the CSV with the CFD simulation results
df = pd.read_csv('data/train_dataset_ready.csv')

# Apply scaling to the concentration data
df['C_in_scaled'] = df['C_in'] * SCALE
df['C_out_scaled'] = df['C_out'] * SCALE

# Extract features for downstream computation
V_in = df['V_in'].values        # Inlet velocity
Area = df['Area'].values        # Cross-sectional area
Dist = df['Distance'].values    # Distance
C_in_scaled = df['C_in_scaled'].values  # Scaled inlet concentration
C_out_scaled = df['C_out_scaled'].values # Scaled outlet concentration (ground truth)

# ==========================================
# 2. Block A: hypothesis formula fitting
# ==========================================
print("Running the block-A fit...")

# Define the physical hypothesis formula
# Physics-motivated: concentration decays with distance, modulated by velocity and area
def hypothesis_formula(X, a, b, c, d):
    v, area, dist, c_in = X
    # Timescale term: sqrt(distance / velocity) - characteristic diffusion time
    time_scale = np.sqrt(dist / (v + 1e-6))
    # Effective-area term: diffusion range corrected by the timescale
    effective_term = area - b * time_scale + c
    # Denominator: distance combined with effective area - similar to diffusion-equation solutions
    # np.maximum Guard against physically meaningless values when the effective area goes negative
    denominator = (a * dist) / (np.maximum(effective_term, 0.1)) + d
    return c_in / denominator

# Initial guess for the fit
p0 = [0.08, 0.6, 26.0, 1.0] 
# Pack the independent variables
X_data = (V_in, Area, Dist, C_in_scaled)

try:
    # Fit the parameters via non-linear least squares
    # bounds Constrain parameter ranges to avoid non-physical / too-large values
    popt, pcov = curve_fit(hypothesis_formula, X_data, C_out_scaled, p0=p0, 
                           maxfev=20000, bounds=([0, 0, 0, 0.5], [2.0, 10.0, 100.0, 2.0]))
    # Compute the fitted predictions
    y_pred_fit = hypothesis_formula(X_data, *popt)
    # Compute R^2 as the goodness-of-fit measure
    r2_fit = r2_score(C_out_scaled, y_pred_fit)
    print("Block A fit succeeded.")
except Exception as e:
    print(f"Block A fit failed: {e}")
    exit()

# ==========================================
# 3. Block B: residual analysis
# ==========================================
print("Running the block-B residual analysis...")

# Use a known second parameter set (from literature or a prior iteration)
a_2, b_2, c_2, d_2 = 9.851695, 10.058320, 34.511347, -14.437245

# Operate on the raw-scale data (variables renamed here to avoid collisions)
C_in_raw = df['C_in'].values * SCALE 
C_true_raw = df['C_out'].values * SCALE

# Compute predictions under the second model
effective_area_2 = Area + c_2 * np.sqrt(V_in) + d_2
denominator_2 = (Dist / effective_area_2) + b_2
C_pred_2 = C_in_raw * (a_2 / denominator_2)

# Compute residuals (truth - prediction)
residuals_2 = C_true_raw - C_pred_2
# Standardised residuals (residual / std) - |value| > 2 is typically an outlier
std_residuals_2 = residuals_2 / np.std(residuals_2)
# R^2 of this model
r2_2 = r2_score(C_true_raw, C_pred_2)

# ==========================================
# 4. Combined visualisation
# ==========================================
print("Rendering the combined figure...")

# Create a 22x12-inch canvas
fig = plt.figure(figsize=(22, 12)) 
# Split the canvas via GridSpec into a left/right pair (ratio 1.1:1)
outer_gs = gridspec.GridSpec(1, 2, width_ratios=[1.1, 1], wspace=0.15)

# --- Left panel: fit curves for four representative cases ---
# 2x2 Sub-panel layout
left_gs = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=outer_gs[0], hspace=0.25, wspace=0.2)
np.random.seed(42) # Fix the RNG for reproducibility
# Draw 4 cases for display
sample_cases = np.random.choice(df['Case'].unique(), 4, replace=False)

unit_c = r"($10^{-7} \cdot \text{kg/m}^3$)"

for i, case_id in enumerate(sample_cases):
    ax = fig.add_subplot(left_gs[i // 2, i % 2])
    
    # Only add this to the top-left sub-panel '(a)' label
    if i == 0:
        ax.text(-0.15, 1.15, '(a)', transform=ax.transAxes, 
                fontsize=26, fontweight='bold', va='top', ha='right', color=VizConfig.COLOR_AXIS)

    # Slice out the current case's data and sort by distance
    case_data = df[df['Case'] == case_id].sort_values('Distance')
    # Compute the predicted curve from the fitted parameters
    pred_c = hypothesis_formula((case_data['V_in'].values, case_data['Area'].values, 
                                 case_data['Distance'].values, case_data['C_in_scaled'].values), *popt)
    
    # Plot the CFD data points (deep blue)
    ax.plot(case_data['Distance'].values, case_data['C_out_scaled'].values, 'o', 
            color=VizConfig.COLOR_MAIN, markersize=5, 
            alpha=0.7, label='CFD Data', rasterized=True, markeredgecolor='white', markeredgewidth=0.5)
    
    # Plot the fit curve (red)
    ax.plot(case_data['Distance'].values, pred_c, 
            color=VizConfig.COLOR_HIGHLIGHT, linewidth=2.5, label='Proposed Formula')
    
    # Set sub-panel titles and axes labels
    ax.set_title(f"Case: {case_id} (V={case_data['V_in'].values[0]:.2f}m/s, Area={case_data['Area'].values[0]:.1f}$m^2$)", 
                 fontsize=VizConfig.LABEL_SIZE, pad=12)
    ax.set_xlabel("Distance (m)", fontsize=VizConfig.LABEL_SIZE, labelpad=8)
    ax.set_ylabel(f"Concentration {unit_c}", fontsize=VizConfig.LABEL_SIZE, labelpad=8)
    ax.tick_params(axis='both', labelsize=VizConfig.TICK_SIZE)
    ax.legend(fontsize=VizConfig.LEGEND_SIZE)
    ax.grid(True, alpha=0.3)

# --- Right panel: statistical analysis (parity + residuals) ---
right_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[1], hspace=0.35)

# --- Top-right: parity plot ---
# Shows how closely predictions track ground truth
ax_parity = fig.add_subplot(right_gs[0])

# add '(b)' label
ax_parity.text(-0.1, 1.1, '(b)', transform=ax_parity.transAxes, 
               fontsize=26, fontweight='bold', va='top', ha='right', color=VizConfig.COLOR_AXIS)

# Scatter (truth vs prediction)
ax_parity.plot(C_out_scaled, y_pred_fit, 'o', color=VizConfig.COLOR_MAIN, markersize=3, alpha=0.2, rasterized=True)
# Draw the y=x reference line
ax_parity.plot([C_out_scaled.min(), C_out_scaled.max()], [C_out_scaled.min(), C_out_scaled.max()], 
               linestyle='--', color=VizConfig.COLOR_HIGHLIGHT, linewidth=2)

ax_parity.set_xlabel(f"True Concentration {unit_c}", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax_parity.set_ylabel(f"Predicted Concentration {unit_c}", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax_parity.set_title(f"Parity Plot ($R^2={r2_fit:.4f}$)", fontsize=VizConfig.TITLE_SIZE, pad=15, fontweight='bold')
ax_parity.tick_params(labelsize=VizConfig.TICK_SIZE)
ax_parity.grid(True, alpha=0.3)

# --- Bottom-right: residual plot ---
# Inspect heteroscedasticity and model bias
ax_resid = fig.add_subplot(right_gs[1])

# add '(c)' label
ax_resid.text(-0.1, 1.1, '(c)', transform=ax_resid.transAxes, 
              fontsize=26, fontweight='bold', va='top', ha='right', color=VizConfig.COLOR_AXIS)

# Scatter the standardised residuals (purple)
ax_resid.plot(C_pred_2, std_residuals_2, 'o', color=VizConfig.COLOR_PALETTE[5], markersize=5, alpha=0.2, rasterized=True, markeredgecolor='none') 

# Reference lines: y=0 and +/- 2 sigma thresholds
ax_resid.axhline(0, color=VizConfig.COLOR_AXIS, linestyle='--', linewidth=1.5)
ax_resid.axhline(2, color=VizConfig.COLOR_WARNING, linestyle=':', linewidth=2)
ax_resid.axhline(-2, color=VizConfig.COLOR_WARNING, linestyle=':', linewidth=2)

ax_resid.set_xlabel(r"Predicted Concentration ($10^7 \cdot \text{kg/m}^3$)", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax_resid.set_ylabel("Standardized Residuals (Dimensionless)", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax_resid.set_title(f"Residual Plot (Check for Heteroscedasticity)\n$R^2 = {r2_2:.4f}$", 
                   fontsize=VizConfig.TITLE_SIZE, fontweight='bold', pad=15)
ax_resid.tick_params(labelsize=VizConfig.TICK_SIZE)
ax_resid.grid(True, alpha=0.3)

# ==========================================
# 5. Save output
# ==========================================
# tight_layout to avoid overlap; pad_inches controls the margin
# tight_layout Works despite the warning in complex GridSpec cases
# You can also call fig.subplots_adjust() manually if needed
# plt.tight_layout() 
output_path = os.path.join(OUTPUT_DIR, "1.pdf")
plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0.1) 

print(f"\nLabelled combined figure saved to: {output_path}")
plt.show()